In [123]:
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
llm = fr"{project_root}\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\2024 NBKB 203_v1.3.html"
verified = fr"{project_root}\data\Documents_Annotés\2024 NBKB 203_LLMv1.3_Verified_EG.htmL"

with open(llm, "r", encoding="utf-8") as f:
    llm_html = f.read()
with open(verified, "r", encoding="utf-8") as f:
    verified_html = f.read()

In [124]:
import re
from typing import Dict, List, Tuple, Optional
from bs4 import BeautifulSoup, NavigableString, Tag

def extract_body(html_content: str) -> str:
    """
    Extract only the body content from HTML, excluding style, script, and head tags.
    Returns the exact string representation of the <body> element to keep reversibility.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    body = soup.find('body')
    if body is not None:
        return str(body)
    print("   ⚠ Warning: No <body> tag found, returning original content")
    return html_content
def tokenize(html_body: str, print=False) -> list:
    """
    Split HTML into a sequence of tokens that preserves:
    - HTML tags as single tokens (e.g., '<div class="x">')
    - Whitespace runs as separate tokens (spaces, newlines, tabs)
    - Punctuation as separate tokens (e.g., ',', '.', ';')
    - Words as separate tokens
    This ensures decode(tokens) == html_body with simple join AND prevents
    merging of words with punctuation after tag removal.
    """
    # Enhanced pattern: separates tags, whitespace, punctuation, words, and other chars
    # Group 1: HTML tags
    # Group 2: Whitespace runs
    # Group 3: Common punctuation (as separate tokens)
    # Group 4: Word characters (alphanumeric + underscore)
    # Group 5: Any other single character
    pattern = re.compile(r"(<[^>]*>)|(\s+)|([.,;:!?()[\]{}\"'`‑–—])|(\w+)|([^\w\s<>])")
    tokens = []
    for m in pattern.finditer(html_body):
        tok = m.group(1) or m.group(2) or m.group(3) or m.group(4) or m.group(5)
        if tok:  # Safety check
            tokens.append(tok)

    if print:
        print(f"   ✓ Tokenized into {len(tokens)} tokens (tags+whitespace+punctuation+words, reversible)")
    return tokens


def decode(tokens: list, print=False) -> str:
    """
    Reconstruct HTML body by concatenating tokens exactly.
    """
    reconstructed_html = "".join(tokens)

    if print:
        print(f"   ✓ Decoded {len(tokens)} tokens into HTML")
    return reconstructed_html


def is_tag(token: str) -> bool:
    return token.startswith("<") and token.endswith(">")


def is_opening_label(token: str) -> Tuple[bool, str]:
    """Returns (is_label, tag_type) where tag_type is 'manual_label' or 'auto_label'."""
    for tag_type in ("manual_label", "auto_label"):
        # Match <manual_label ...> or <manual_label> (with or without attributes)
        if re.match(rf"<{tag_type}(\s|>)", token):
            return True, tag_type
    return False, ""


def is_closing_label(token: str, tag_type: str) -> bool:
    return token == f"</{tag_type}>"


def parse_label_attributes(token: str) -> Tuple[str, Dict[str, str]]:
    """
    Extract labelname and other attributes from an opening label tag token.
    e.g. '<manual_label labelname="FOO" color="red">'
    """
    labelname = ""
    attributes = {}

    for m in re.finditer(r'(\w+)=["\']([^"\']*)["\']', token):
        key, val = m.group(1), m.group(2)
        if key == "labelname":
            labelname = val
        elif key != "style":
            attributes[key] = val

    return labelname, attributes

In [125]:
from bs4 import BeautifulSoup
from collections import defaultdict
from typing import Dict, List, Tuple
import os
from pathlib import Path
import re
from datetime import datetime



from typing import Dict, List, Tuple

class Span:
    def __init__(self, text: str, start: int, end: int, labelname: str,
                 attributes: Dict[str, str], context_text: str = "", type: str = "manual_label"):
        self.text = text.strip()
        self.start = start
        self.end = end
        self.labelname = labelname
        self.attributes = attributes
        self.context_text = context_text
        self.type = type

    def __repr__(self):
        return f"Span('{self.text[:30]}...', label={self.labelname}, start={self.start})"


def extract_spans_from_html(html_body: str, context_chars: int = 200) -> Tuple[str, List["Span"]]:
    """
    Tokenize the HTML body once, then walk token by token:
    - Tag tokens are skipped when building plain text (they don't contribute characters)
    - Non-tag tokens advance the plain-text offset exactly
    - When an opening label tag is encountered, we snapshot the current offset
    - When the matching closing tag is encountered, we know start and end precisely
    - Context is a slice of the plain text around [start, end] — no searching at all

    Handles nesting: a label inside another label works correctly because
    we track a stack. The outer label's end is recorded when *its* closing
    tag is seen, by which point the inner label has already been fully processed.
    """
    tokens = tokenize(html_body)

    # --- Pass 1: build the plain text string and a token→char-offset map ---
    # plain_offset[i] = the plain-text offset BEFORE token i is consumed.
    # For tag tokens the offset does not advance (they contribute 0 chars).
    plain_chars: List[str] = []
    token_start_offset: List[int] = []  # plain-text offset at the START of each token

    for tok in tokens:
        token_start_offset.append(len(plain_chars))
        if not is_tag(tok):
            plain_chars.extend(tok)  # extend char by char to keep offsets exact

    plain_text = "".join(plain_chars)
    # Sentinel: offset after the last token
    token_start_offset.append(len(plain_text))

    # --- Pass 2: walk tokens, detect labels, emit Spans ---
    spans: List["Span"] = []

    # Stack entries: (tag_type, labelname, attributes, plain_text_start, token_index_of_open)
    stack: List[Tuple[str, str, Dict, int]] = []

    for i, tok in enumerate(tokens):
        is_lbl, tag_type = is_opening_label(tok)

        if is_lbl:
            labelname, attributes = parse_label_attributes(tok)
            plain_start = token_start_offset[i]  # offset right before this tag
            stack.append((tag_type, labelname, attributes, plain_start))

        elif is_tag(tok) and stack:
            # Check if this closes the innermost open label
            top_tag_type = stack[-1][0]
            if is_closing_label(tok, top_tag_type):
                tag_type, labelname, attributes, plain_start = stack.pop()
                # plain-text offset right after this closing tag = same as start of next token
                plain_end = token_start_offset[i]  # closing tag itself adds 0 chars

                span_text = plain_text[plain_start:plain_end]
                normalized_text = " ".join(span_text.split())

                if normalized_text:
                    ctx_start = max(0, plain_start - context_chars)
                    ctx_end = min(len(plain_text), plain_end + context_chars)
                    context_text = plain_text[ctx_start:ctx_end]

                    spans.append(Span(
                        text=normalized_text,
                        start=plain_start,
                        end=plain_end,
                        labelname=labelname,
                        attributes=attributes,
                        context_text=context_text,
                        type=tag_type,
                    ))

    return plain_text, spans

In [129]:
_, spans1 = extract_spans_from_html(html_body=extract_body(llm_html), context_chars=200)
_, spans2 = extract_spans_from_html(html_body=extract_body(verified_html), context_chars=200)
print(f"\n✓ LLM: {len(spans1)} spans")
print(f"✓ Verified: {len(spans2)} spans")


✓ LLM: 584 spans
✓ Verified: 657 spans


In [130]:
s2_manual_label = [s for s in spans2 if s.type == "manual_label"]
s2_auto_label = [s for s in spans2 if s.type == "auto_label"]
print(f"\n✓ Verified: {len(s2_manual_label)} manual-labeled spans")
print(f"✓ Verified: {len(s2_auto_label)} auto-labeled spans")


✓ Verified: 195 manual-labeled spans
✓ Verified: 462 auto-labeled spans


In [181]:
matched = []
matched_idx = []
for i, s1 in enumerate(spans1):

    for j, s2 in enumerate(spans2):

        if j in [k for _, k in matched_idx]:
            continue

        if s1.type != s2.type:
            continue

        if s1.labelname != s2.labelname:
            continue

        if s1.text != s2.text:
            continue

        if not jaccard_context_overlap(s1, s2, threshold= 0.9):
            continue

        matched.append((s1, s2))
        matched_idx.append((i, j))

        break

print(f"\n✓ Exact matches (same text, same label type): {len(matched)} spans")


✓ Exact matches (same text, same label type): 462 spans


In [182]:
non_matched_auto_spans = [s1 for i, s1 in enumerate(spans1) if i not in [k for k, _ in matched_idx]]

In [183]:
for s1 in non_matched_auto_spans:
    print("====================")
    print(s1.text)
    print("----------")
    print(s1.context_text)
    print("====================\n\n\n\n")

para. 186
----------
 duty to enter into and conduct
those negotiations in good faith… Ultimately it is through negotiated
settlements…reinforced by the judgments of this Court, that we will achieve…
reconciliation…” (at para. 186). 
 
4.           
As will be discussed
herein, recent decisions from the Supreme Court now direct that the duty to
negotiate land claims is indeed a legal one and not just a moral one: see
Haida Nat




at para. 186
----------
al, duty to enter into and conduct
those negotiations in good faith… Ultimately it is through negotiated
settlements…reinforced by the judgments of this Court, that we will achieve…
reconciliation…” (at para. 186). 
 
4.           
As will be discussed
herein, recent decisions from the Supreme Court now direct that the duty to
negotiate land claims is indeed a legal one and not just a moral one: see
Haida Nat




Lax Kw'alaams Indian Band v. Canada (Attorney General)
----------
parties fair notice of the
case to meet, provide the bound

In [193]:
import re


def jaccard_context_overlap(span1, span2, threshold: float = 0.7) -> bool:
    """
    Check approximate overlap between two spans using Jaccard similarity
    over the union of their context_text and text words (case-insensitive).
    """
    ctx1 = f"{span1.context_text} {span1.text}".lower() if span1.context_text is not None else span1.text.lower()
    ctx2 = f"{span2.context_text} {span2.text}".lower() if span2.context_text is not None else span2.text.lower()
    
    words1 = set(ctx1.split())
    words2 = set(ctx2.split())
    
    if not words1 or not words2:
        return False
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    if union == 0:
        return False
    
    similarity = intersection / union
    return similarity >= threshold

def share_text_word(span1, span2) -> bool:
    """
    Return True if the two spans share at least one word in their *text* fields.
    Used to define overlap once context is approximately the same.
    Now robust to trailing punctuation like commas/periods (e.g. 'ewert' vs 'ewert,').
    """
    # Normalize by stripping punctuation and extracting word characters only
    words1 = set(re.findall(r"\w+", span1.text.lower()))
    words2 = set(re.findall(r"\w+", span2.text.lower()))
    
    if not words1 or not words2:
        return False
    
    return len(words1 & words2) > 0


removed_auto_no_manual = []
auto_with_manual_candidate = []

for n_s1 in non_matched_auto_spans:
    found_overlap = False

    for s2 in s2_manual_label:
        
        
        # 1) same approximate context
        if not jaccard_context_overlap(n_s1, s2, threshold=0.9):
            continue
        
        # 2) at least one common word in span.text
        if share_text_word(n_s1, s2):
            auto_with_manual_candidate.append((n_s1, s2))
            found_overlap = True
            break
        
    
    # If we never found a manual span with same approx context AND
    # at least one shared word in the text, then this auto is truly removed.
    if not found_overlap:
        #print("--------------------------------------------------------")
        #print("not matched : truly removed")
        #print(n_s1.text, n_s1.labelname, n_s1.type)
        #print(n_s1.context_text)
        removed_auto_no_manual.append(n_s1)


print(f"Non-matched auto spans: {len(non_matched_auto_spans)}")
print(f"→ With likely manual replacement (overlap): {len(auto_with_manual_candidate)}")
print(f"→ Truly removed (no overlap under same approx context): {len(removed_auto_no_manual)}")

print("\nExamples of truly removed auto spans:")
for span in removed_auto_no_manual:
    print("-", span.text[:120].replace("\n", " "), f"[label={span.labelname}, type={span.type}]")

Non-matched auto spans: 122
→ With likely manual replacement (overlap): 96
→ Truly removed (no overlap under same approx context): 26

Examples of truly removed auto spans:
- This article [label=title, type=auto_label]
- decision [label=title, type=auto_label]
- At issue in this Claim, is whether that radical title was and is burdened by Aboriginal title, deemed an “independent le [label=decision, type=auto_label]
- 23.01 [label=fragment, type=auto_label]
- Where Available [label=title, type=auto_label]
- (1) [label=fragment, type=auto_label]
- 23.01Where Available (1) [label=legislation, type=auto_label]
- [emphasis added] [label=secondary sources, type=auto_label]
- 1.02.1 [label=fragment, type=auto_label]
- 1.02.1 [label=legislation, type=auto_label]
- 1.03 [label=fragment, type=auto_label]
- (2) [label=fragment, type=auto_label]
- 1.03(2) [label=legislation, type=auto_label]
- Ewert [label=decision, type=auto_label]
- Manitoba Metis [label=decision, type=auto_label]
- AT [label=tit

In [178]:
for s2 in s2_manual_label:
    print("====================")
    print(s2.text)
    print("----------")
    print(s2.context_text)
    print("====================\n\n\n\n")

Wolastoqey Nations v. New Brunswick and Canada, et.al.
----------


Wolastoqey Nations v.
New Brunswick and Canada, et.al., 2024 NBKB 203
FC-322-2021
 
IN THE COURT OF KING’S BENCH OF NEW BRUNSWICK
TRIAL DIVISION
JUDICIAL DISTRICT OF FREDERICTON
 
B
E T W E E N:
WOLASTOQEY NATION AT
WELAMUKOTUK (OROMOCTO FIRST NATION), W




2024 NBKB 203
----------


Wolastoqey Nations v.
New Brunswick and Canada, et.al., 2024 NBKB 203
FC-322-2021
 
IN THE COURT OF KING’S BENCH OF NEW BRUNSWICK
TRIAL DIVISION
JUDICIAL DISTRICT OF FREDERICTON
 
B
E T W E E N:
WOLASTOQEY NATION AT
WELAMUKOTUK (OROMOCTO FIRST NATION), WOLASTOQEY NATIO




Wolastoqey Nations v. New Brunswick and Canada, et.al., 2024 NBKB 203
----------


Wolastoqey Nations v.
New Brunswick and Canada, et.al., 2024 NBKB 203
FC-322-2021
 
IN THE COURT OF KING’S BENCH OF NEW BRUNSWICK
TRIAL DIVISION
JUDICIAL DISTRICT OF FREDERICTON
 
B
E T W E E N:
WOLASTOQEY NATION AT
WELAMUKOTUK (OROMOCTO FIRST NATION), WOLASTOQEY NATIO




para. 186
-----

In [190]:
count=0
for spans in removed_auto_no_manual:
    for s2 in s2_manual_label:
        
        
        # 1) same approximate context
        if not jaccard_context_overlap(spans, s2, threshold=0.9):
            continue
        

        # 2) at least one common word in span.text
        words1 = set(spans.text.lower().split())
        words2 = set(s2.text.lower().split())

        print(words1)
        print(words2)
        print(len(words1 & words2))

        if share_text_word(spans, s2):
            auto_with_manual_candidate.append((spans, s2))
            found_overlap = True
            break

{'decision'}
{'tsilhqot’in'}
0


In [194]:
# Build a clear, formatted summary report of the verification process
print("\n================ Annotation Verification Report ================")

# Compute derived sets if needed
s2_manual_label = [s for s in spans2 if s.type == "manual_label"]
s2_auto_label = [s for s in spans2 if s.type == "auto_label"]
total_verified_spans = len(spans2) 

modified_auto = len(auto_with_manual_candidate)

new_manual = len(s2_manual_label) - modified_auto
removed_auto = len(removed_auto_no_manual)

# Percentages relative to the verified document
def pct(part, whole):
    return (part / whole * 100) if whole else 0
pct_modified_auto = pct(modified_auto, total_verified_spans)
pct_new_manual = pct(new_manual, total_verified_spans)
pct_removed_auto = pct(removed_auto, total_verified_spans)

report = f"""
First document (entirely annotated by an LLM) has {len(spans1)} auto-labeled spans,
while the verified version has {len(spans2)} labeled spans.

In the verified document there are {len(s2_manual_label)} manual-labeled spans
and {len(s2_auto_label)} auto-labeled spans.

There are {modified_auto} modified auto_label spans
({pct_modified_auto:.1f}% of spans in the verified document).

Hence there are {new_manual} new manual-labeled spans that were not in the original LLM annotation
({pct_new_manual:.1f}% of spans in the verified document).

Hence there are {removed_auto} auto-labeled spans that were removed during the verification process
({pct_removed_auto:.1f}% of spans in the verified document).
"""

print(report)
print("===============================================================\n")


================ Annotation Verification Report ================

First document (entirely annotated by an LLM) has 584 auto-labeled spans,
while the verified version has 657 labeled spans.

In the verified document there are 195 manual-labeled spans
and 462 auto-labeled spans.

There are 96 modified auto_label spans
(14.6% of spans in the verified document).

Hence there are 99 new manual-labeled spans that were not in the original LLM annotation
(15.1% of spans in the verified document).

Hence there are 26 auto-labeled spans that were removed during the verification process
(4.0% of spans in the verified document).




In [196]:
for s1, s2 in auto_with_manual_candidate:
    print("====================")
    print(s1.text)
    print(s2.text)

para. 186
para. 186
at para. 186
para. 186
Lax Kw'alaams Indian Band v. Canada (Attorney General)
Lax Kw'alaams Indian Band v. Canada (Attorney General)
2011 SCC 56
2011 SCC 56
para. 43
para. 43
Lax Kw'alaams Indian Band v. Canada (Attorney General), 2011 SCC 56, at para. 43
Lax Kw'alaams Indian Band v. Canada (Attorney General)
Rule 37
Rule 37
Tsilhqot’in decision
Tsilhqot’in
Tsilhqot’in
Tsilhqot’in, at para. 69
Tsilhqot’in,
Tsilhqot’in, at para. 20
Tsilhqot’in,
Tsilhqot’in, at para. 21-23
Indigenous Law Journal. Volume 8, Issue 1 (2010)
Indigenous Law Journal. Volume 8, Issue 1 (2010), p. 7-26
p. 7-26
Indigenous Law Journal. Volume 8, Issue 1 (2010), p. 7-26
R. v.
R. v. Marshall
Marshall
R. v. Marshall
Rule 23
Rule 23
Rule 23
Rule 23
Rule 23
Rule 23
Tingley
Tingley, at para 112
Rule 23(1)(b)
Rule 23(1)(b)
With respect to Rule 23(1)(b)
Rule 23(1)(b)
Rule 23
Rule 23
Rule 23
Rule 23
Rule 22
Rule 22
Rule 22
Rule 22
Rule 23
Rule 23
Rule 23
Rule 23
Sewell v. ING Insurance, 2007 NBCA 42
Sew

In [197]:
only_in_s1 = [s1 for s1, _ in auto_with_manual_candidate]
only_in_s2 = [s2 for _, s2 in auto_with_manual_candidate]

print(sum(len(s.text) for s in only_in_s1)/len(only_in_s1))
print(sum(len(s.text) for s in only_in_s2)/len(only_in_s2))

20.083333333333332
22.53125


In [202]:
import difflib
from collections import Counter

def analyze_edge_modifications(s1_text: str, s2_text: str):
    """
    Use the existing `tokenize` function to compare two span texts token-by-token
    and characterize edge changes going from s1 → s2 as:
      - extend_left / reduce_left / same / complex_left
      - extend_right / reduce_right / same / complex_right
    """
    # Tokenize and drop pure-whitespace tokens (spaces/newlines are not informative here)
    t1 = [t for t in tokenize(s1_text) if not t.isspace()]
    t2 = [t for t in tokenize(s2_text) if not t.isspace()]
    
    sm = difflib.SequenceMatcher(a=t1, b=t2)
    blocks = sm.get_matching_blocks()
    if len(blocks) == 0:
        return {
            "left_change": "no_common_subsequence",
            "right_change": "no_common_subsequence",
            "left_removed": t1,
            "left_added": t2,
            "right_removed": [],
            "right_added": [],
        }
    
    # First and last *real* blocks (last element is a 0-length sentinel)
    first = blocks[0]
    last = blocks[-2] if len(blocks) > 1 else blocks[0]
    
    # Left side analysis
    # first.a = index in t1 where the first common block starts
    # first.b = index in t2 where the first common block starts
    if first.a == 0 and first.b == 0:
        left_change = "same"
    elif first.a > 0 and first.b == 0:
        # Extra tokens at the left in s1 that disappeared in s2
        left_change = "reduce_left"
    elif first.a == 0 and first.b > 0:
        # Extra tokens at the left in s2 that were not in s1
        left_change = "extend_left"
    else:
        # Both sides have unmatched prefixes → more complex than pure extend/reduce
        left_change = "complex_left"
    
    left_removed = t1[:first.a] if first.a > 0 else []
    left_added = t2[:first.b] if first.b > 0 else []
    
    # Right side analysis
    end1 = len(t1)
    end2 = len(t2)
    a_end = last.a + last.size
    b_end = last.b + last.size
    
    if a_end == end1 and b_end == end2:
        right_change = "same"
    elif a_end < end1 and b_end == end2:
        # Extra tokens at the right in s1 that disappeared in s2
        right_change = "reduce_right"
    elif a_end == end1 and b_end < end2:
        # Extra tokens at the right in s2 that were not in s1
        right_change = "extend_right"
    else:
        right_change = "complex_right"
    
    right_removed = t1[a_end:] if a_end < end1 else []
    right_added = t2[b_end:] if b_end < end2 else []
    
    return {
        "left_change": left_change,
        "right_change": right_change,
        "left_removed": left_removed,
        "left_added": left_added,
        "right_removed": right_removed,
        "right_added": right_added,
    }

# Apply to all (s1, s2) pairs in auto_with_manual_candidate
edge_change_counts = Counter()

for idx, (s1, s2) in enumerate(zip(only_in_s1, only_in_s2), start=1):
    res = analyze_edge_modifications(s1.text, s2.text)
    key = (res["left_change"], res["right_change"])
    edge_change_counts[key] += 1
    
    # Print a few illustrative examples
    if idx <= 10:
        print("====================")
        print(f"Pair {idx}")
        print("s1:", s1.text)
        print("s2:", s2.text)
        print("Texts equal:", s1.text == s2.text)
        print("Left change:", res["left_change"],"| removed:", res["left_removed"],"| added:", res["left_added"])
        print("Right change:", res["right_change"],"| removed:", res["right_removed"],"| added:", res["right_added"])

print("\n===== Summary of edge modifications (s1 → s2) =====")
for (left, right), count in edge_change_counts.items():
    print(f"Left={left:14s} | Right={right:14s} : {count}")

Pair 1
s1: para. 186
s2: para. 186
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 2
s1: at para. 186
s2: para. 186
Texts equal: False
Left change: reduce_left | removed: ['at'] | added: []
Right change: same | removed: [] | added: []
Pair 3
s1: Lax Kw'alaams Indian Band v. Canada (Attorney General)
s2: Lax Kw'alaams Indian Band v. Canada (Attorney General)
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 4
s1: 2011 SCC 56
s2: 2011 SCC 56
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 5
s1: para. 43
s2: para. 43
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 6
s1: Lax Kw'alaams Indian Band v. Canada (Attorney General), 2011 SCC 56, at para. 43
s2: Lax Kw'alaams Indian Band v. Canada (Attorney General)
Texts equal: False
Left change: 

Left=same           | Right=same           : 43
Left=reduce_left    | Right=same           : 9
Left=same           | Right=reduce_right   : 7
Left=same           | Right=extend_right   : 32
Left=extend_left    | Right=same           : 5